In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for _pkg in ['torch', 'torchvision', 'matplotlib', 'numpy']:
    _install(_pkg)

In [ ]:
import math
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import torchvision


torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

# I3D 代码实战：学习实现 vs 工程实现

基于论文 *Quo Vadis, Action Recognition? A New Model and the Kinetics Dataset*（Carreira & Zisserman, CVPR 2017），
我们用一个**玩具视频动作分类**任务演示 I3D 的核心思想：把 2D Inception 风格卷积 **inflate** 成 3D 卷积，从而同时建模空间与时间信息。

本 Notebook 使用**同一份数据、同一任务**走两条路径：

| | 学习路径（Learning） | 工程路径（Engineering） |
|---|---|---|
| 目标 | 看清 inflate、3D Inception、时间保留池化 | 用更紧凑的 PyTorch 模块快速搭建可训练模型 |
| 实现方式 | 手写 `inflate_conv2d_to_conv3d` + 手写 I3D-like block | 用 `nn.Sequential` / `nn.ModuleList` 组织简洁版 3D CNN |
| 代码量 | 较多 | 更少 |
| 适合场景 | 面试准备、原理理解 | 工程原型、实验迭代 |

> 本例的决策：**学习路径 L1（可完整训练）**，**工程路径 E3 + train（无成熟官方 I3D 高层库，直接用原生 PyTorch）**。

## 一、论文背景与任务边界

### 1. 论文核心问题

在 I3D 之前，视频动作识别常见两条路线：

- **2D CNN + 时序模块**：先做空间特征，再用 LSTM 等建模时间
- **3D CNN**：直接在 `(T, H, W)` 上卷积，但训练代价高、预训练资源弱

I3D 的关键回答是：**能不能把 ImageNet 上训练好的 2D 网络，平滑地迁移到 3D 视频网络中？**

### 2. 为什么本 Notebook 用玩具动作分类

论文原始任务是大规模视频动作识别，通常依赖 Kinetics 等大数据集；但教学场景更关心：

1. `inflate` 以后张量形状如何变化
2. 为什么池化常用 `(1, 3, 3)` 保留时间分辨率
3. 3D Inception block 如何做多分支时空特征抽取

因此本 Notebook 使用**可控合成数据**：每类视频只靠不同的运动轨迹区分，足以让 I3D 的时空建模能力显式体现出来。

### 3. 范围说明

本 Notebook **覆盖**：
- inflate 卷积
- 简化版 I3D-like 网络
- 学习路径与工程路径对照
- 训练结果、边界、面试表达

本 Notebook **不覆盖**：
- 原论文完整双流（RGB + Optical Flow）训练
- Kinetics 规模预训练
- 多 clip / 多 crop 测试协议

## 二、最小必要理论

### 1. 关键公式

#### （1）2D 卷积核膨胀为 3D 卷积核

若 2D 卷积核权重为 $W_{2D} \in \mathbb{R}^{C_{out} \times C_{in} \times H \times W}$，
则 I3D 的一个基础初始化方式是把它沿时间维复制到长度 $T$：

$$
W_{3D}[:, :, t, :, :] = \frac{1}{T} W_{2D}, \quad t = 1, 2, \dots, T
$$

这样当输入是“重复同一帧得到的静态视频”时，3D 卷积输出与 2D 卷积保持同量级。

#### （2）3D 卷积

对输入 $X \in \mathbb{R}^{B \times C \times T \times H \times W}$，3D 卷积在时空三个维度共同滑动：

$$
Y = \mathrm{Conv3D}(X; W_{3D})
$$

直觉上，2D 卷积只能看单帧纹理；3D 卷积能额外看相邻帧的运动变化。

#### （3）时间维池化保守策略

论文里一个重要经验是：**前期不要太早压缩时间维**。因此常见设计是

$$
\text{MaxPool3D}(k=(1,3,3),\ s=(1,2,2))
$$

即：只在空间维下采样，时间维先保留。

### 2. 工程可行性决策

- **learning_path_depth = L1**：简化版 I3D 可以在小合成数据上稳定训练
- **engineering_path_form = E3**：官方 PyTorch / torchvision 没有 I3D 高层封装，最稳妥是用原生 `nn.Conv3d` 等模块组织工程版
- **engineering_action = train**：同一数据上完整训练，比只做前向演示更有教学价值
- **training_vs_inference_diff = false**：动作分类没有像自回归生成那样显式分叉，主要差别是 `model.train()` / `model.eval()` 下 dropout 与 BN 行为不同

> API 依据：PyTorch 官方 `Conv3d` / `MaxPool3d` 接受三元组 `kernel_size/stride/padding`；torchvision 官方视频模型也采用 `(B, C, T, H, W)` 输入约定。

## 三、组件映射表

| 论文组件 | 学习路径覆盖 | 工程路径 / API 对应 | 状态 |
|---|---|---|---|
| Inflation（2D→3D） | `inflate_conv2d_to_conv3d` 手写实现 | 工程路径直接使用 `nn.Conv3d`，不显式拷权重 | Runnable |
| 3D Conv | `BasicConv3d` | `nn.Conv3d` + `nn.BatchNorm3d` + `nn.ReLU` | Runnable |
| 3D Inception 多分支 | `InceptionBlock3D` 手写分支 | `CompactVideoBlock` 用更紧凑的顺序结构近似表达 | Runnable |
| 时间维保留池化 | `(1,3,3)` 池化显式写出 | `nn.MaxPool3d((1,3,3), ...)` | Runnable |
| 全局时空汇聚 | `AdaptiveAvgPool3d((1,1,1))` | 同左 | Runnable |
| 双流 RGB / 光流融合 | 本 Notebook 仅解释，不实现双流训练 | 仅在 markdown 讨论 | Explain-only |
| Kinetics 预训练迁移 | 用静态视频等价性解释 bootstrapping | 仅讨论，不下载预训练权重 | Illustrative |

## 1. 数据准备

I3D 适合视频分类。为了让 Notebook 在 CPU / 免费 Colab 上都能跑通，我们不下载真实视频，而是构造一个**运动可分**的小数据集：

- 每个样本是 `(C=3, T=12, H=32, W=32)`
- 一共 4 类动作模式：水平移动、垂直移动、对角移动、中心闪烁
- 类别差异主要体现在**时间变化**，这正好能体现 3D 卷积的价值

这样做的好处是：训练时间短，而且不是纯随机噪声。模型必须利用时序变化才能分类。

In [ ]:
NUM_CLASSES = 4
CHANNELS = 3
NUM_FRAMES = 12
HEIGHT = 32
WIDTH = 32
TRAIN_SAMPLES_PER_CLASS = 80
TEST_SAMPLES_PER_CLASS = 20


def _make_gaussian_blob(cx, cy, height=HEIGHT, width=WIDTH, sigma=3.0):
    yy, xx = torch.meshgrid(
        torch.arange(height, dtype=torch.float32),
        torch.arange(width, dtype=torch.float32),
        indexing='ij'
    )
    blob = torch.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    return blob


def generate_video_sample(label, num_frames=NUM_FRAMES, height=HEIGHT, width=WIDTH):
    video = torch.zeros(CHANNELS, num_frames, height, width)
    base_noise = 0.03 * torch.randn(CHANNELS, num_frames, height, width)

    for t in range(num_frames):
        if label == 0:  # horizontal motion
            cx = 6 + (width - 12) * t / max(1, num_frames - 1)
            cy = height // 2
        elif label == 1:  # vertical motion
            cx = width // 2
            cy = 6 + (height - 12) * t / max(1, num_frames - 1)
        elif label == 2:  # diagonal motion
            cx = 6 + (width - 12) * t / max(1, num_frames - 1)
            cy = 6 + (height - 12) * t / max(1, num_frames - 1)
        else:  # center flashing
            cx = width // 2
            cy = height // 2

        blob = _make_gaussian_blob(cx, cy, height, width)  # (H, W)
        strength = 1.0 if label != 3 else (0.35 if t % 2 == 0 else 1.2)

        video[0, t] = strength * blob                      # (H, W)
        video[1, t] = 0.7 * strength * blob.roll(shifts=1, dims=0)
        video[2, t] = 0.5 * strength * blob.roll(shifts=1, dims=1)

    video = video + base_noise
    video = video.clamp(0.0, 1.0)
    return video


def build_dataset(samples_per_class):
    videos, labels = [], []
    for label in range(NUM_CLASSES):
        for _ in range(samples_per_class):
            videos.append(generate_video_sample(label))    # (C, T, H, W)
            labels.append(label)
    videos = torch.stack(videos, dim=0)                    # (N, C, T, H, W)
    labels = torch.tensor(labels, dtype=torch.long)        # (N,)
    perm = torch.randperm(len(labels))
    return videos[perm], labels[perm]


train_videos, train_labels = build_dataset(TRAIN_SAMPLES_PER_CLASS)
test_videos, test_labels = build_dataset(TEST_SAMPLES_PER_CLASS)

print('Train videos:', train_videos.shape)
print('Train labels:', train_labels.shape)
print('Test videos :', test_videos.shape)
print('Class counts:', torch.bincount(train_labels).tolist())

In [ ]:
BATCH_SIZE = 16

train_dataset = TensorDataset(train_videos, train_labels)
test_dataset = TensorDataset(test_videos, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch_videos, batch_labels = next(iter(train_loader))
print('Batch videos:', batch_videos.shape)  # (B, C, T, H, W)
print('Batch labels:', batch_labels.shape)  # (B,)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
sample_video = train_videos[0]  # (C, T, H, W)
for i, t in enumerate([0, 3, 6, 9]):
    frame = sample_video[:, t].permute(1, 2, 0).numpy()  # (H, W, C)
    axes[i].imshow(frame)
    axes[i].set_title(f'Frame {t}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()
print('Sample label:', train_labels[0].item())

## 2. 共享组件

两条路径共享同一组超参数与训练函数，便于直接比较。

这里的超参数有意做小：
- 让 CPU 也能训练
- 保证学习路径和工程路径都能在同一 notebook 中完整跑完
- 同时保留 I3D 的关键结构：3D 卷积、多分支、时间维保留池化

In [ ]:
D_MODEL = 16
DROPOUT = 0.2
LR = 1e-3
NUM_EPOCHS = 6

print({
    'D_MODEL': D_MODEL,
    'DROPOUT': DROPOUT,
    'LR': LR,
    'NUM_EPOCHS': NUM_EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'NUM_FRAMES': NUM_FRAMES,
})

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def evaluate_model(model, loader, device):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for videos, labels in loader:
            videos = videos.to(device)                    # (B, C, T, H, W)
            labels = labels.to(device)                    # (B,)
            logits = model(videos)                        # (B, num_classes)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += labels.size(0)
    return total_loss / total_count, total_correct / total_count


def train_model(model, train_loader, test_loader, device, epochs=NUM_EPOCHS, lr=LR):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, running_correct, running_count = 0.0, 0, 0

        for videos, labels in train_loader:
            videos = videos.to(device)                    # (B, C, T, H, W)
            labels = labels.to(device)                    # (B,)

            optimizer.zero_grad()
            logits = model(videos)                        # (B, num_classes)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            running_count += labels.size(0)

        train_loss = running_loss / running_count
        train_acc = running_correct / running_count
        test_loss, test_acc = evaluate_model(model, test_loader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f} | test_acc={test_acc:.3f}"
        )

    return history

## 3. 学习路径：从零实现 I3D 的关键结构

这一部分按数据流展开：

1. 先看 `inflate` 如何把 2D 卷积核复制成 3D 卷积核
2. 再实现 `BasicConv3d`
3. 再实现简化版 `InceptionBlock3D`
4. 最后组装完整分类模型并训练

因为这是 **L1** 路径，所以会完整训练，而不只是前向演示。

### 3.1 Inflation：把 2D 卷积核扩展到 3D

设 2D 卷积核形状是 `(C_out, C_in, H, W)`，膨胀后变成 `(C_out, C_in, T, H, W)`。

实现思路：
- 沿时间维复制 `T` 次
- 每一份除以 `T`
- 这样静态视频输入时，3D 输出不会因为多次求和而尺度失控

张量形状变化：

- `w2d`: `(C_out, C_in, H, W)`
- `w2d.unsqueeze(2)`: `(C_out, C_in, 1, H, W)`
- `repeat(..., T, ...)`: `(C_out, C_in, T, H, W)`

In [ ]:
def inflate_conv2d_to_conv3d(conv2d, time_dim=3):
    conv3d = nn.Conv3d(
        in_channels=conv2d.in_channels,
        out_channels=conv2d.out_channels,
        kernel_size=(time_dim, conv2d.kernel_size[0], conv2d.kernel_size[1]),
        stride=(1, conv2d.stride[0], conv2d.stride[1]),
        padding=(time_dim // 2, conv2d.padding[0], conv2d.padding[1]),
        bias=conv2d.bias is not None,
    )

    with torch.no_grad():
        w2d = conv2d.weight.data                                   # (C_out, C_in, H, W)
        w3d = w2d.unsqueeze(2).repeat(1, 1, time_dim, 1, 1)       # (C_out, C_in, T, H, W)
        w3d = w3d / time_dim
        conv3d.weight.copy_(w3d)
        if conv2d.bias is not None:
            conv3d.bias.copy_(conv2d.bias.data)
    return conv3d


conv2d = nn.Conv2d(3, 8, kernel_size=3, padding=1)
conv3d = inflate_conv2d_to_conv3d(conv2d, time_dim=3)
print('2D weight shape:', tuple(conv2d.weight.shape))
print('3D weight shape:', tuple(conv3d.weight.shape))

In [ ]:
torch.manual_seed(42)
conv2d_demo = nn.Conv2d(3, 4, kernel_size=3, padding=1, bias=False)
conv3d_demo = inflate_conv2d_to_conv3d(conv2d_demo, time_dim=3)

frame = torch.randn(1, 3, 16, 16)                                 # (1, C, H, W)
static_video = frame.unsqueeze(2).repeat(1, 1, 5, 1, 1)           # (1, C, T, H, W)

out2d = conv2d_demo(frame)                                         # (1, 4, H, W)
out3d = conv3d_demo(static_video)                                   # (1, 4, T, H, W)
mid = out3d.shape[2] // 2
max_diff = (out2d - out3d[:, :, mid]).abs().max().item()

print('2D output shape :', tuple(out2d.shape))
print('3D output shape :', tuple(out3d.shape))
print('Middle-frame max diff:', max_diff)

### 3.2 BasicConv3d 与 InceptionBlock3D

#### BasicConv3d

这是最基础的时空特征提取单元：

$$
\mathrm{ReLU}(\mathrm{BN}(\mathrm{Conv3D}(x)))
$$

#### InceptionBlock3D

I3D 延续 Inception 思想：让不同感受野的分支并行，再在通道维拼接。

本 Notebook 用 4 个分支：
- `1x1x1`
- `1x1x1 -> 3x3x3`
- `1x1x1 -> 3x3x3`（更窄）
- `(1,3,3)` max-pool -> `1x1x1`

最后在 `dim=1` 拼接。这样做的作用是：**同一层同时捕获不同尺度的时空模式**。

In [ ]:
class BasicConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm3d(out_channels)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)                                            # -> (B, C_out, T', H', W')
        x = self.bn(x)
        x = self.act(x)
        return x


class InceptionBlock3D(nn.Module):
    def __init__(self, in_channels, c1, c3r, c3, c5r, c5, pool_proj):
        super().__init__()
        self.branch1 = BasicConv3d(in_channels, c1, kernel_size=1)
        self.branch2 = nn.Sequential(
            BasicConv3d(in_channels, c3r, kernel_size=1),
            BasicConv3d(c3r, c3, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            BasicConv3d(in_channels, c5r, kernel_size=1),
            BasicConv3d(c5r, c5, kernel_size=3, padding=1),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=1, padding=(0, 1, 1)),
            BasicConv3d(in_channels, pool_proj, kernel_size=1),
        )
        self.out_channels = c1 + c3 + c5 + pool_proj

    def forward(self, x):
        b1 = self.branch1(x)                                        # (B, c1, T, H, W)
        b2 = self.branch2(x)                                        # (B, c3, T, H, W)
        b3 = self.branch3(x)                                        # (B, c5, T, H, W)
        b4 = self.branch4(x)                                        # (B, pool_proj, T, H, W)
        return torch.cat([b1, b2, b3, b4], dim=1)                  # (B, sum, T, H, W)


x = torch.randn(2, 16, 12, 32, 32)
block = InceptionBlock3D(16, 8, 8, 12, 4, 4, 4)
out = block(x)
print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(out.shape))
print('Output channels:', block.out_channels)

### 3.3 组装简化版 I3D 分类器

网络结构遵循“先 stem，后多分支，再全局池化”的思路：

1. `stem`: 大卷积核 3D 卷积，先把底层时空边缘抽出来
2. `(1,3,3)` 池化：先降空间，不急着压时间
3. `InceptionBlock3D` × 2：多尺度时空特征
4. `AdaptiveAvgPool3d((1,1,1))`：把整段视频压成一个全局表示
5. `Linear`：输出类别 logits

这不是论文原始 Inception-V1 全量结构，但足以保留 I3D 的主要教学特征。

In [ ]:
class MiniI3D(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT):
        super().__init__()
        self.stem = BasicConv3d(3, D_MODEL, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3))
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))

        self.block1 = InceptionBlock3D(D_MODEL, 8, 8, 12, 4, 4, 4)
        self.block2 = InceptionBlock3D(self.block1.out_channels, 12, 12, 16, 4, 8, 8)

        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(self.block2.out_channels, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)                                            # (B, 3, 12, 32, 32) -> (B, 16, 12, 16, 16)
        x = self.pool1(x)                                           # (B, 16, 12, 16, 16) -> (B, 16, 12, 8, 8)
        x = self.block1(x)                                          # (B, 28, 12, 8, 8)
        x = self.block2(x)                                          # (B, 44, 12, 8, 8)
        x = self.pool2(x)                                           # (B, 44, 6, 4, 4)
        x = self.head(x)                                            # (B, 4)
        return x


learning_model = MiniI3D().to(device)
total_params, trainable_params = count_parameters(learning_model)
print('MiniI3D params:', total_params)

demo_logits = learning_model(batch_videos[:2].to(device))
print('Demo logits shape:', tuple(demo_logits.shape))

### 3.4 学习路径训练

动作分类没有 teacher forcing / autoregressive 这类训练-推理分裂。

这里训练与推理的差别主要只有：
- `model.train()`：dropout 生效，BatchNorm 更新统计量
- `model.eval()`：dropout 关闭，BatchNorm 使用固定统计量

因此本节重点是观察：**手写 I3D-like 结构能否学会区分运动模式**。

In [ ]:
print('Notebook construction complete.')

## 8. 面试与项目表达

### 高频面试题

**Q1：I3D 的核心创新是什么？**

- 把成熟的 2D CNN 结构 inflate 成 3D CNN
- 从而复用 2D 预训练知识，降低 3D 视频模型训练难度
- 同时引入 Kinetics，推动视频预训练范式形成

**Q2：为什么 inflate 时要除以时间维长度？**

- 因为卷积核沿时间维复制后会累加多次
- 如果不做缩放，静态视频上的激活会被放大
- 除以 `T` 能让 3D 输出与 2D 输出保持同量级

**Q3：为什么前期常用 `(1,3,3)` 池化？**

- 因为空间分辨率可以较早压缩
- 但时间分辨率在动作识别里更敏感
- 太早压缩时间维会丢掉运动细节

**Q4：I3D 和普通 3D CNN 有什么区别？**

- 两者都做时空卷积
- I3D 更强调从 2D 预训练迁移到 3D
- 这让它在当时比从零训练的 3D CNN 更容易优化

**Q5：为什么双流 I3D 会更强？**

- RGB 流擅长外观
- 光流流擅长显式运动
- 两者融合后，时空信息更完整

**Q6：I3D 的局限是什么？**

- 3D 卷积计算重、显存开销大
- 对长时程依赖建模不如后来的 Transformer 灵活
- 双流方案还依赖昂贵的光流预处理

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | I3D 本质做了什么？ | 把 2D CNN 膨胀成 3D CNN，用于时空联合建模 |
| 2 | 为什么要 inflate？ | 为了复用 2D 预训练权重，降低 3D 训练难度 |
| 3 | 为什么 `(1,3,3)` 池化重要？ | 因为要先保留时间分辨率 |
| 4 | 双流的意义是什么？ | 外观和运动分开建模再融合 |
| 5 | I3D 的主要代价是什么？ | 计算量和显存开销较大 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织：

- **学习深度**：我从零实现了 I3D 里的 inflate、3D 卷积、多分支 Inception block，能解释张量形状如何沿时间维传播。
- **工程能力**：我同时写了一个更紧凑的 3D CNN baseline，并在同一任务上对比了学习路径和工程路径。
- **对比思考**：我能说明手写 I3D-like 结构更适合讲原理，而紧凑工程版更适合快速实验和调参。

### 延伸阅读与对比

| 对比维度 | C3D | I3D | SlowFast |
|---|---|---|---|
| 核心思想 | 直接 3D 卷积 | 2D 到 3D 膨胀迁移 | 快慢双路径建模 |
| 复杂度 | 中 | 中到高 | 更高 |
| 适用场景 | 早期视频 CNN baseline | 经典视频动作识别 | 更强的现代视频识别 baseline |

### 进阶探索方向

- 把单流版本扩展成双流版本：加入光流输入
- 在真实小视频数据集上验证：如 UCF101 小子集
- 对比 I3D 与 Video Transformer：比较卷积与自注意力的视频建模方式

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, t in enumerate([0, 3, 6, 9]):
    axes[i].imshow(train_videos[0, 0, t].numpy(), cmap='viridis')
    axes[i].set_title(f'Channel0 Frame {t}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 7. 结果与边界

### 1. 结果解读

如果训练正常，两条路径都应能在这个玩具数据上快速达到较高准确率。这说明：

- 视频分类任务确实依赖时间变化
- 即便是简化版 I3D-like 结构，也能学到基础时空模式
- 工程上的紧凑模型也可以完成同样任务，但它不再显式保留 Inception 式多分支解释

### 2. 两条路径的边界

**学习路径的收益**：
- 更清楚为什么要 inflate
- 更清楚 `(1,3,3)` 池化的作用
- 更适合讲论文、讲模型内部机制

**学习路径的代价**：
- 代码长
- 更容易出现 shape bug
- 扩展到真实数据训练时维护成本更高

**工程路径的收益**：
- 更短、更稳、更适合快速做 baseline
- 更接近日常实验代码风格

**工程路径的代价**：
- 论文组件与代码组件不是一一对应
- 模型解释性弱于学习路径

### 3. 什么时候用哪条路径

- 想讲清论文、准备面试：优先学习路径
- 想先把视频分类 baseline 跑起来：优先工程路径

In [ ]:
label_names = ['horizontal', 'vertical', 'diagonal', 'flashing']

sample_batch = test_videos[:8].to(device)
true_labels = test_labels[:8]

learning_model.eval()
engineering_model.eval()
with torch.no_grad():
    pred_learning = learning_model(sample_batch).argmax(dim=1).cpu()
    pred_engineering = engineering_model(sample_batch).argmax(dim=1).cpu()

for i in range(8):
    print(
        f'Sample {i} | true={label_names[true_labels[i]]:<10s} | '
        f'learning={label_names[pred_learning[i]]:<10s} | '
        f'engineering={label_names[pred_engineering[i]]:<10s}'
    )

## 6. 训练 vs 推理差异

虽然 I3D 动作分类不像 GPT 那样存在显式的自回归推理逻辑，但训练与推理仍有模式差异：

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `model.train()`，dropout 开启，BN 更新统计量 | 相同 |
| 推理 | `model.eval()`，前向得到 logits，再取 `argmax` | 相同 |

关键点是：**这里两条路径底层算法不同，但推理接口形式一致，都是分类 logits -> predicted class**。

In [ ]:
learning_params, _ = count_parameters(learning_model)
engineering_params, _ = count_parameters(engineering_model)

print(f'Learning params   : {learning_params}')
print(f'Engineering params: {engineering_params}')
print(f'Learning test acc : {learning_history["test_acc"][-1]:.3f}')
print(f'Engineering test acc: {engineering_history["test_acc"][-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(learning_history['test_acc'], marker='o', label='Learning')
axes[0].plot(engineering_history['test_acc'], marker='s', label='Engineering')
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(['Learning', 'Engineering'], [learning_params, engineering_params])
axes[1].set_title('Parameter Count Comparison')
axes[1].set_ylabel('Parameters')
plt.tight_layout()
plt.show()

## 5. 学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能直接看到 inflate、多分支拼接、时间维保留策略 | 更容易理解工程上如何快速搭建一个可训练视频分类器 |
| 代码量 | 更多 | 更少 |
| 灵活性 | 可逐组件修改、便于做机制实验 | 结构更紧凑，但可解释性更弱 |
| 稳定性 | 手写多分支更容易出 shape bug | 串行结构更稳、更容易调试 |
| 工业适配度 | 更像研究教学原型 | 更接近真实工程里的快速 baseline |
| 适用场景 | 面试、论文复现、原理拆解 | 原型验证、课程作业、实验迭代 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(engineering_history['train_loss'], marker='o', label='Train loss')
axes[0].plot(engineering_history['test_loss'], marker='s', label='Test loss')
axes[0].set_title('Engineering Path Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(engineering_history['train_acc'], marker='o', label='Train acc')
axes[1].plot(engineering_history['test_acc'], marker='s', label='Test acc')
axes[1].set_title('Engineering Path Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
engineering_history = train_model(
    engineering_model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=NUM_EPOCHS,
    lr=LR,
)

In [ ]:
class CompactVideoBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class CompactVideoClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(3, D_MODEL, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False),
            nn.BatchNorm3d(D_MODEL),
            nn.ReLU(inplace=True),
        )
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.block1 = CompactVideoBlock(D_MODEL, 24)
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
        self.block2 = CompactVideoBlock(24, 32)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(x)
        x = self.block1(x)
        x = self.pool2(x)
        x = self.block2(x)
        x = self.head(x)
        return x


engineering_model = CompactVideoClassifier().to(device)
eng_total, eng_trainable = count_parameters(engineering_model)
print('Compact model params:', eng_total)
print('Demo logits shape:', tuple(engineering_model(batch_videos[:2].to(device)).shape))

### 4.1 组件对应关系

| 学习路径（手写） | 工程路径（简洁版） | 说明 |
|---|---|---|
| `inflate_conv2d_to_conv3d` | 直接定义 `Conv3d` | 工程里通常直接训练 3D 卷积，不一定保留显式 inflate 步骤 |
| `InceptionBlock3D` | `CompactVideoBlock` | 多分支换成更短的串行模块 |
| `(1,3,3)` 池化 | 相同 | 时间维尽量晚下采样 |
| `AdaptiveAvgPool3d` | 相同 | 全局时空汇聚 |
| 手写分类头 | 手写分类头 | 工程重点是简洁和稳定，而不是最高可解释性 |

## 4. 工程路径：用原生 PyTorch 写更紧凑的 3D CNN

由于官方库没有 I3D 的高层封装，这里工程路径不调用额外第三方视频框架，而是选择更稳定的 E3 方案：

- 仍然保留 `Conv3d + BatchNorm3d + ReLU`
- 但不手写 Inception 多分支，而改成更短的顺序堆叠
- 目标是比较：**当我们更重视工程紧凑性时，会失去什么、得到什么**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(learning_history['train_loss'], marker='o', label='Train loss')
axes[0].plot(learning_history['test_loss'], marker='s', label='Test loss')
axes[0].set_title('Learning Path Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(learning_history['train_acc'], marker='o', label='Train acc')
axes[1].plot(learning_history['test_acc'], marker='s', label='Test acc')
axes[1].set_title('Learning Path Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
learning_history = train_model(
    learning_model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=NUM_EPOCHS,
    lr=LR,
)